In [ ]:
from google.colab import files
uploaded = files.upload()

Saving legal_contract_clauses.xlsx to legal_contract_clauses.xlsx


In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import tensorflow as tf

In [ ]:
df = pd.read_excel('legal_contract_clauses.xlsx')

# Fix column names
df.columns = ['text', 'type', 'risk']

# Remove incorrect header row
df = df.iloc[1:].reset_index(drop=True)

# Drop missing values
df = df.dropna(subset=['text', 'type', 'risk'])

# Remove duplicates
df = df.drop_duplicates()

df.head()

,text,type,risk
0,Electric City of Illinois L.L.C.,Parties,low
1,The term of this Agreement shall be ten (10)...,Effective Date,low
2,Unless earlier terminated otherwise prov...,Effective Date,high
3,If Distributor comp...,Renewal Term,low
4,This Agreement is to be construed according to...,Governing Law,low


In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]
    return " ".join(words)

df['clean_text'] = df['text'].apply(preprocess_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 9000
max_len = 100

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(df['clean_text'])

sequences = tokenizer.texts_to_sequences(df['clean_text'])

X = pad_sequences(sequences, maxlen=max_len)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_type = LabelEncoder()
le_risk = LabelEncoder()

y_type = le_type.fit_transform(df['type'])
y_risk = le_risk.fit_transform(df['risk'])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_type_train, y_type_test = train_test_split(
    X, y_type, test_size=0.2, random_state=42
)

_, _, y_risk_train, y_risk_test = train_test_split(
    X, y_risk, test_size=0.2, random_state=42
)

In [ ]:
from tensorflow.keras.layers import Layer

class PositionalEncoding(Layer):
    def __init__(self, max_len, embed_dim):
        super(PositionalEncoding, self).__init__()

        pos = np.arange(max_len)[:, np.newaxis]
        i = np.arange(embed_dim)[np.newaxis, :]

        angle_rates = 1 / np.power(10000, (2 * (i//2)) / np.float32(embed_dim))
        angle_rads = pos * angle_rates

        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

        self.pos_encoding = tf.cast(angle_rads[np.newaxis, ...], dtype=tf.float32)

    def call(self, inputs):
        return inputs + self.pos_encoding[:, :tf.shape(inputs)[1], :]

In [ ]:
class MultiHeadSelfAttention(Layer):
    def __init__(self, embed_dim, num_heads=2):
        super().__init__()
        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

    def call(self, inputs):
        return self.mha(inputs, inputs)

In [ ]:
from tensorflow.keras.layers import Dense, Dropout, LayerNormalization

class TransformerBlock(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()

        self.att = MultiHeadSelfAttention(embed_dim, num_heads)

        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])

        self.norm1 = LayerNormalization()
        self.norm2 = LayerNormalization()

        self.dropout1 = Dropout(0.1)
        self.dropout2 = Dropout(0.1)

    def call(self, inputs):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output)

        out1 = self.norm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)

        return self.norm2(out1 + ffn_output)

In [ ]:
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D
from tensorflow.keras.models import Model

embed_dim = 64
num_heads = 2
ff_dim = 64

inputs = Input(shape=(max_len,))

x = Embedding(vocab_size, embed_dim)(inputs)
x = PositionalEncoding(max_len, embed_dim)(x)

transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
x = transformer_block(x)

x = GlobalAveragePooling1D()(x)
x = Dropout(0.1)(x)

# Outputs
type_output = Dense(len(set(y_type)), activation="softmax", name="type")(x)
risk_output = Dense(len(set(y_risk)), activation="softmax", name="risk")(x)

model = Model(inputs=inputs, outputs=[type_output, risk_output])

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 100, 64)   │    576,000 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encodin… │ (None, 100, 64)   │          0 │ embedding_1[0][0] │
│ (PositionalEncodin… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_block_1 │ (None, 100, 64)   │     41,792 │ positional_encod… │
│ (TransformerBlock)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ transformer_bloc… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 64)        │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ type (Dense)        │ (None, 41)        │      2,665 │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ risk (Dense)        │ (None, 3)         │        195 │ dropout_7[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 620,652 (2.37 MB)

 Trainable params: 620,652 (2.37 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer="adam",
    loss={
        "type": "sparse_categorical_crossentropy",
        "risk": "sparse_categorical_crossentropy"
    },
    metrics={
        "type": "accuracy",
        "risk": "accuracy"
    }
)

In [ ]:
history = model.fit(
    X_train,
    {"type": y_type_train, "risk": y_risk_train},
    epochs=7,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/7
213/213 ━━━━━━━━━━━━━━━━━━━━ 23s 110ms/step - loss: 0.7361 - risk_accuracy: 0.9747 - risk_loss: 0.0846 - type_accuracy: 0.8212 - type_loss: 0.6522 - val_loss: 1.3571 - val_risk_accuracy: 0.9101 - val_risk_loss: 0.2905 - val_type_accuracy: 0.7368 - val_type_loss: 1.0539
Epoch 2/7
213/213 ━━━━━━━━━━━━━━━━━━━━ 41s 110ms/step - loss: 0.5468 - risk_accuracy: 0.9818 - risk_loss: 0.0585 - type_accuracy: 0.8665 - type_loss: 0.4882 - val_loss: 1.4391 - val_risk_accuracy: 0.9193 - val_risk_loss: 0.2885 - val_type_accuracy: 0.7156 - val_type_loss: 1.1375
Epoch 3/7
213/213 ━━━━━━━━━━━━━━━━━━━━ 41s 110ms/step - loss: 0.4227 - risk_accuracy: 0.9871 - risk_loss: 0.0456 - type_accuracy: 0.8987 - type_loss: 0.3770 - val_loss: 1.6310 - val_risk_accuracy: 0.8968 - val_risk_loss: 0.4164 - val_type_accuracy: 0.7222 - val_type_loss: 1.2091
Epoch 4/7
213/213 ━━━━━━━━━━━━━━━━━━━━ 40s 106ms/step - loss: 0.3303 - risk_accuracy: 0.9884 - risk_loss: 0.0374 - type_accuracy: 0.9230 - type_loss: 0.2930 - v

In [ ]:
pred_type, pred_risk = model.predict(X_test)
# for i in range(5):
#   print(X_test[i])
pred_type_labels = np.argmax(pred_type, axis=1)
pred_risk_labels = np.argmax(pred_risk, axis=1)

60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step
[   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0  198
   34  291  344    4 3695  342  304  159 2626 4816  802  342  304  318
  205    7]
[   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0   33  746   33   50
    3 1697    2   65  831

In [ ]:
decoded_types = le_type.inverse_transform(pred_type_labels)
decoded_risks = le_risk.inverse_transform(pred_risk_labels)

print(decoded_types[:5])
print(decoded_risks[:5])

['Source Code Escrow' 'Insurance' 'Document Name' 'Anti-Assignment'
 'License Grant']
['medium' 'medium' 'low' 'high' 'low']


In [ ]:
def predict_clause(text):
    clean = preprocess_text(text)
    seq = tokenizer.texts_to_sequences([clean])
    padded = pad_sequences(seq, maxlen=max_len)

    pred_type, pred_risk = model.predict(padded)

    type_label = le_type.inverse_transform([np.argmax(pred_type)])[0]
    risk_label = le_risk.inverse_transform([np.argmax(pred_risk)])[0]

    return {"Type": type_label, "Risk": risk_label}

#print(predict_clause("The agreement will terminate if obligations are breached"))
#print(predict_clause("[Party B] shall pay the invoice within [15] days of receipt. Late payments shall not accrue interest unless agreed upon in writing. Invoices shall be accompanied by a brief description of services completed."))
#print(predict_clause( "Both parties agree to keep all commercial and technical information shared under this agreement confidential and not disclose it to any third party for a period of [1] year after termination, except as required by law."))
print(predict_clause("A minimum of a $250,000.00  purchase  order  must be  received  by Company by the first of each month for a total (12) month  period."))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
{'Type': 'Minimum Commitment', 'Risk': 'medium'}


In [ ]:
test_clauses = [
    "This agreement shall terminate immediately upon the occurrence of a material breach by either party, and the non-breaching party shall have the right to pursue all available legal remedies.",

    "The employee agrees to maintain strict confidentiality of all proprietary and sensitive information obtained during the course of employment and shall not disclose such information to any third party without prior written consent.",

    "The client shall make payment for all services rendered within thirty (30) days from the date of invoice, without any deductions or penalties unless otherwise agreed in writing.",

    "The vendor shall be held fully liable for any direct, indirect, or consequential damages arising due to negligence, misconduct, or failure to fulfill contractual obligations under this agreement.",

    "Either party may terminate this agreement by providing a written notice of at least thirty (30) days to the other party, without assigning any reason for such termination.",

    "The lessee shall not sublease, assign, or transfer the leased premises to any third party without obtaining prior written consent from the lessor.",

    "Any disputes, claims, or controversies arising out of or relating to this agreement shall be resolved through binding arbitration in accordance with the applicable arbitration laws.",

    "The company shall not be held liable for any indirect, incidental, or consequential damages, including loss of profits or business interruption, arising out of the use of its services.",

    "The contractor agrees to perform the services in accordance with the agreed specifications, timelines, and quality standards as outlined in this agreement.",

    "Failure to comply with applicable data protection and privacy regulations shall result in immediate termination of this agreement and may lead to legal action and financial penalties."
]

for clause in test_clauses:
    print("Clause:\n", clause)
    print("Prediction:", predict_clause(clause))
    print("="*80)

Clause:
 This agreement shall terminate immediately upon the occurrence of a material breach by either party, and the non-breaching party shall have the right to pursue all available legal remedies.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step
Prediction: {'Type': 'Termination For Convenience', 'Risk': 'high'}
Clause:
 The employee agrees to maintain strict confidentiality of all proprietary and sensitive information obtained during the course of employment and shall not disclose such information to any third party without prior written consent.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 293ms/step
Prediction: {'Type': 'Ip Ownership Assignment', 'Risk': 'high'}
Clause:
 The client shall make payment for all services rendered within thirty (30) days from the date of invoice, without any deductions or penalties unless otherwise agreed in writing.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step
Prediction: {'Type': 'Post-Termination Services', 'Risk': 'high'}
Clause:
 The vendor shall be held fully liable for any dire

In [ ]:
print("Number of contract types:", len(le_type.classes_))

Number of contract types: 41


In [ ]:
def split_into_clauses(contract_text):
    # Simple split based on period
    clauses = contract_text.split(".")

    # Remove empty clauses
    clauses = [c.strip() for c in clauses if len(c.strip()) > 20]

    return clauses

In [ ]:
def analyze_contract(contract_text):
    clauses = split_into_clauses(contract_text)

    results = []

    for clause in clauses:
        prediction = predict_clause(clause)
        results.append({
            "clause": clause,
            "type": prediction["Type"],
            "risk": prediction["Risk"]
        })

    return results

In [ ]:
contract_text = """This Service Agreement is entered into between the Company and the Contractor.
The contractor agrees to provide services as per the agreed specifications and timelines outlined in this agreement.

The contractor shall maintain confidentiality of all proprietary and sensitive information obtained during the course of this engagement and shall not disclose such information to any third party.

The client shall make payment within thirty (30) days from the date of invoice without any deductions unless agreed otherwise.

Either party may terminate this agreement by providing a written notice of thirty (30) days. In case of breach of contract, termination may be immediate.

The contractor shall be liable for any damages arising due to negligence or failure to perform the agreed services.

Any disputes arising out of this agreement shall be resolved through arbitration in accordance with applicable laws.

Failure to comply with applicable regulations may result in termination of this agreement and legal action.
"""

results = analyze_contract(contract_text)

for r in results:
    print("Clause:", r["clause"])
    print("Type:", r["type"])
    print("Risk:", r["risk"])
    print("-"*60)



# from collections import Counter

# def overall_contract_type(results):
#     types = [r["type"] for r in results]

#     type_count = Counter(types)

#     # Get most common type
#     final_type = type_count.most_common(1)[0][0]

#     return final_type

# print("Final Contract Type:", overall_contract_type(results))


def weighted_contract_type(results):
    weights = {
        "Liability": 3,
        "Termination": 3,
        "Compliance": 2
    }

    score = {}

    for r in results:
        t = r["type"]
        w = weights.get(t, 1)  # default weight = 1

        score[t] = score.get(t, 0) + w

    return max(score, key=score.get)

print("Weighted Contract Type:", weighted_contract_type(results))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
Clause: This Service Agreement is entered into between the Company and the Contractor
Type: Expiration Date
Risk: low
------------------------------------------------------------
Clause: The contractor agrees to provide services as per the agreed specifications and timelines outlined in this agreement
Type: Exclusivity
Risk: medium
------------------------------------------------------------
Clause: The contractor shall maintain confidentiality of all proprietary and sensitive information obtained during the course of this engagement and shall not disclose such information to any third party
Type: Joint Ip Ownership
Risk: medium
-------------------